# BirdCLEF 2026 — BirdSet EfficientNet-B1 Benchmark
Same pipeline as the B0 baseline, swapping in the bioacoustic-pretrained BirdSet EfficientNet-B1 backbone. Identical data/val/metric so scores are comparable.

## 0. Install & Imports

In [ ]:
# BirdSet EfficientNetB1 is a bioacoustic-pretrained checkpoint hosted on HF hub,
# loaded through timm. transformers is used to pull the checkpoint.
!pip install -q timm transformers

In [ ]:
import os, math, random, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
import timm

warnings.filterwarnings("ignore")

## 1. Config

In [ ]:
CFG = {
    "seed": 42,
    # paths — same layout as the EDA notebook
    "base": Path("/content/drive/MyDrive/Kaggle/birdclef-2026"),
    # audio
    "sr": 32000,
    "segment_sec": 5,
    "n_mels": 128,
    "n_fft": 2048,
    "hop_length": 512,
    "fmin": 20,
    "fmax": 16000,
    "in_chans": 3,          # repeat mel to 3ch to keep pretrained stem weights
    # model
    "backbone": "efficientnet_b1",
    "model_tag": "birdset_b1",
    "birdset_ckpt": "DBD-research-group/EfficientNet-B1-BirdSet-XCL",
    "pretrained": True,
    # training
    "batch_size": 32,
    "epochs": 10,
    "lr": 1e-3,
    "weight_decay": 1e-2,
    "val_fraction": 0.2,   # fraction of labeled soundscape FILES held out for val
    "num_workers": 2,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG["seed"])

BASE                    = CFG["base"]
TRAIN_AUDIO             = BASE / "train_audio"
TRAIN_SOUNDSCAPES       = BASE / "train_soundscapes"
TRAIN_SOUNDSCAPE_LABELS = BASE / "train_soundscapes_labels.csv"
TRAIN_META              = BASE / "train.csv"
SAMPLE_SUB              = BASE / "sample_submission.csv"

## 2. Scored species (from submission columns)

In [ ]:
# Scored species come from the submission columns, NOT from train metadata.
sample_sub = pd.read_csv(SAMPLE_SUB)
SPECIES = [c for c in sample_sub.columns if c != "row_id"]
NUM_CLASSES = len(SPECIES)
species_to_idx = {s: i for i, s in enumerate(SPECIES)}
print(f"{NUM_CLASSES} scored species")

## 3. Data prep — training pool + soundscape val split
XC/iNat clips are weak whole-recording labels. Labeled soundscapes are the only test-matched data, split by **file** so no recording spans train and val.

In [ ]:
# ---- Training pool: XC/iNat clips (weak, one primary_label per file) ----
meta = pd.read_csv(TRAIN_META)
meta = meta[meta["primary_label"].isin(species_to_idx)].reset_index(drop=True)
meta["path"] = meta["filename"].map(lambda f: str(TRAIN_AUDIO / f))

# ---- Soundscapes: strong per-segment multi-label; the only test-matched data ----
ss = pd.read_csv(TRAIN_SOUNDSCAPE_LABELS)
ss["path"] = ss["filename"].map(lambda f: str(TRAIN_SOUNDSCAPES / f))

# Split soundscapes by FILE (not segment) to avoid within-recording leakage.
gss = GroupShuffleSplit(n_splits=1, test_size=CFG["val_fraction"], random_state=CFG["seed"])
tr_idx, va_idx = next(gss.split(ss, groups=ss["filename"]))
ss_train, ss_val = ss.iloc[tr_idx].reset_index(drop=True), ss.iloc[va_idx].reset_index(drop=True)
print(f"train clips: {len(meta)} | soundscape segs -> train {len(ss_train)}  val {len(ss_val)}")

## 4. Audio → mel spectrogram

In [ ]:
def load_segment(path, sr, seg_len, offset=None, random_crop=False):
    """Load a fixed-length waveform segment, pad/crop as needed."""
    dur = librosa.get_duration(path=path)
    if offset is None:
        max_off = max(0.0, dur - seg_len)
        offset = random.uniform(0, max_off) if random_crop else 0.0
    y, _ = librosa.load(path, sr=sr, offset=offset, duration=seg_len)
    target = int(sr * seg_len)
    if len(y) < target:
        y = np.pad(y, (0, target - len(y)))
    return y[:target]

def wav_to_mel(y, cfg):
    m = librosa.feature.melspectrogram(
        y=y, sr=cfg["sr"], n_fft=cfg["n_fft"], hop_length=cfg["hop_length"],
        n_mels=cfg["n_mels"], fmin=cfg["fmin"], fmax=cfg["fmax"])
    m = librosa.power_to_db(m, ref=np.max)
    m = (m - m.mean()) / (m.std() + 1e-6)
    return m.astype(np.float32)

## 5. Datasets & loaders
Val target is true multi-hot (semicolon codes); clip target is single-positive multi-hot.

In [ ]:
class ClipDataset(Dataset):
    """XC/iNat clips: single primary_label -> multi-hot with one positive."""
    def __init__(self, df, cfg, train=True):
        self.df, self.cfg, self.train = df, cfg, train
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        y = load_segment(r["path"], self.cfg["sr"], self.cfg["segment_sec"],
                         random_crop=self.train)
        x = torch.from_numpy(wav_to_mel(y, self.cfg)).unsqueeze(0)
        x = x.repeat(self.cfg["in_chans"], 1, 1)
        t = torch.zeros(NUM_CLASSES)
        t[species_to_idx[r["primary_label"]]] = 1.0
        return x, t

class SoundscapeDataset(Dataset):
    """Soundscape segments: semicolon-separated codes -> true multi-hot."""
    def __init__(self, df, cfg):
        self.df, self.cfg = df, cfg
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        y = load_segment(r["path"], self.cfg["sr"], self.cfg["segment_sec"],
                         offset=float(r["start"]))
        x = torch.from_numpy(wav_to_mel(y, self.cfg)).unsqueeze(0)
        x = x.repeat(self.cfg["in_chans"], 1, 1)
        t = torch.zeros(NUM_CLASSES)
        for code_ in str(r["primary_label"]).split(";"):
            if code_ in species_to_idx:
                t[species_to_idx[code_]] = 1.0
        return x, t

# Train on XC/iNat + the train portion of soundscapes; validate ONLY on held-out soundscapes.
train_ds = torch.utils.data.ConcatDataset([
    ClipDataset(meta, CFG, train=True),
    SoundscapeDataset(ss_train, CFG),
])
val_ds = SoundscapeDataset(ss_val, CFG)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,
                          num_workers=CFG["num_workers"], pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True)

## 6. Model

In [ ]:
# BirdSet EfficientNetB1: bioacoustic pretraining on the BirdSet XCL corpus.
# We load the pretrained checkpoint, then replace the classifier head to match
# our 234 scored species. If the hub weights are unavailable in the kernel,
# set CFG['pretrained']=False to fall back to a timm-pretrained B1.
from timm.models.hub import download_cached_file
from safetensors.torch import load_file

class BirdModel(nn.Module):
    def __init__(self, cfg, num_classes):
        super().__init__()
        self.backbone = timm.create_model(
            cfg["backbone"], pretrained=False,
            num_classes=num_classes, in_chans=cfg["in_chans"])
        if cfg["pretrained"]:
            self._load_birdset(cfg)
    def _load_birdset(self, cfg):
        try:
            from huggingface_hub import hf_hub_download
            ckpt = hf_hub_download(cfg["birdset_ckpt"], filename="model.safetensors")
            state = load_file(ckpt)
            # Keep only backbone tensors whose shapes match; skip the pretrained head.
            own = self.backbone.state_dict()
            keep = {k: v for k, v in state.items()
                    if k in own and own[k].shape == v.shape and "classifier" not in k}
            own.update(keep)
            self.backbone.load_state_dict(own)
            print(f"loaded {len(keep)}/{len(own)} tensors from BirdSet checkpoint")
        except Exception as e:
            print(f"BirdSet load failed ({e}); using randomly-init backbone")
    def forward(self, x):
        return self.backbone(x)

model = BirdModel(CFG, NUM_CLASSES).to(CFG["device"])

## 7. Loss, optimizer, metric

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"],
                              weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
scaler = torch.cuda.amp.GradScaler(enabled=CFG["device"] == "cuda")

def macro_auc(y_true, y_prob):
    # Macro ROC-AUC over columns that have both classes present in val.
    aucs = []
    for j in range(y_true.shape[1]):
        if y_true[:, j].sum() > 0 and y_true[:, j].sum() < len(y_true):
            aucs.append(roc_auc_score(y_true[:, j], y_prob[:, j]))
    return float(np.mean(aucs)) if aucs else 0.0

## 8. Train / validate

In [ ]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    total, ys, ps = 0.0, [], []
    for x, t in loader:
        x, t = x.to(CFG["device"]), t.to(CFG["device"])
        with torch.set_grad_enabled(train), torch.cuda.amp.autocast(enabled=CFG["device"]=="cuda"):
            out = model(x)
            loss = criterion(out, t)
        if train:
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
        total += loss.item() * len(x)
        if not train:
            ys.append(t.cpu().numpy()); ps.append(torch.sigmoid(out).float().cpu().numpy())
    avg = total / len(loader.dataset)
    if train:
        return avg, None
    return avg, macro_auc(np.vstack(ys), np.vstack(ps))

best_auc = 0.0
for epoch in range(CFG["epochs"]):
    tr_loss, _ = run_epoch(train_loader, train=True)
    va_loss, va_auc = run_epoch(val_loader, train=False)
    scheduler.step()
    print(f"epoch {epoch+1:02d} | train {tr_loss:.4f} | val {va_loss:.4f} | macroAUC {va_auc:.4f}")
    if va_auc > best_auc:
        best_auc = va_auc
        torch.save(model.state_dict(), f"best_{CFG['model_tag']}.pt")
print(f"best macro ROC-AUC: {best_auc:.4f}")